# <font color="steelblue">Despliegue de modelos con Streamlit</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**


**Fecha última edición**: 10/06/2026

**Licencia**: <small><a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a><br /></small>

No olvides hacer una copia si deseas utilizarlo. Al usar estos contenidos, aceptas nuestros términos de uso y nuestra política de privacidad.

---

# Introducción

El cuaderno está pensado para leerse en orden. Los apartados 1 y 2 son de puesta en marcha: hasta que no consigas ver una aplicación en tu navegador, el resto no te servirá de mucho. Los apartados 3 a 7 son la referencia del lenguaje. El apartado 8 construye una aplicación completa paso a paso, y el 9 la publica en internet.

Si ya tienes Streamlit funcionando, puedes saltar directamente al apartado 3.

---

# Índice

| Módulo | Contenido |
|---|---|
| **0** | Qué es Streamlit y cómo piensa |
| **1** | Ejecución en local: instalación, entornos virtuales, el comando `run` |
| **2** | Ejecución desde Google Colab con ngrok |
| **3** | Widgets de entrada |
| **4** | Elementos de salida |
| **5** | Organización de la pantalla |
| **6** | Caché: el concepto crítico |
| **7** | Estado, formularios y flujo de control |
| **8** | Construcción guiada de una aplicación completa |
| **9** | Publicación en internet con GitHub y Streamlit Community Cloud |
| **10** | Buenas prácticas y catálogo de errores |
| **A** | Apéndice: chuleta de referencia |
| **B** | Apéndice: equivalencias con Gradio |


**Antes de comenzar veremos el resultado de una aplicación para la clasificación de flores iris usando un modelo de random forest:**
* **[Aplicación](https://machinelearningbook-bmrr8pvtqwzfgughgseayf.streamlit.app/)**

---

# Módulo 0. Qué es Streamlit y cómo piensa

## 0.1 El problema que resuelve

Tienes un modelo entrenado en un cuaderno. Funciona. Pero para que otra persona lo use tendría que abrir el cuaderno, entender qué celdas ejecutar, no romper nada y saber dónde cambiar los valores de entrada. Eso no es viable con un cliente, un compañero de otro departamento o un tribunal de un trabajo fin de máster.

Lo que necesitas es una página web: una caja donde escribir, un botón, y un resultado. Tradicionalmente eso significaba aprender HTML, CSS, JavaScript y un framework de servidor. Streamlit elimina todo eso: escribes Python y obtienes una página web.

## 0.2 La idea central

Aquí está la clave de todo el framework, y conviene leerla dos veces:

> **Una aplicación de Streamlit es un script de Python normal que se ejecuta entero, de arriba abajo, cada vez que el usuario interactúa con algo.**

No hay funciones que registrar. No hay eventos que conectar. No hay componentes que declarar y luego enlazar. Solo un script que se lee de la primera línea a la última.

Veámoslo con el ejemplo más pequeño posible:

```python
import streamlit as st

nombre = st.text_input("¿Cómo te llamas?")
st.write(f"Hola, {nombre}")
```

Analicemos qué ocurre en cada momento:

**Primera ejecución (el usuario abre la página).** Streamlit ejecuta el script. La línea `st.text_input` hace dos cosas a la vez: dibuja una caja de texto en la página **y** devuelve su contenido actual, que de momento es la cadena vacía `""`. Esa cadena vacía se guarda en la variable `nombre`. La línea siguiente escribe "Hola, ".

**El usuario escribe "Ana" y pulsa Enter.** Streamlit **vuelve a ejecutar el script completo desde la línea 1**. Esta vez, cuando llega a `st.text_input`, la función devuelve `"Ana"`. La variable `nombre` vale ahora `"Ana"`, y `st.write` muestra "Hola, Ana".

La página no se ha "actualizado parcialmente": se ha regenerado entera. Streamlit se encarga por debajo de que el navegador solo repinte lo que ha cambiado, pero desde el punto de vista de tu código, todo se ha vuelto a ejecutar.

## 0.3 Las cuatro consecuencias que debes interiorizar

Este modelo, aparentemente simple, tiene implicaciones que explican casi todo el comportamiento de Streamlit.

### Consecuencia 1: los widgets son valores, no objetos

En la mayoría de frameworks, un control de interfaz es un objeto al que preguntas su valor:

```python
# Así funcionan otros frameworks
caja = CrearCajaDeTexto()
...
valor = caja.obtener_texto()
```

En Streamlit no. La llamada **es** el valor:

```python
edad = st.slider("Edad", 0, 100, 25)
# 'edad' es un número entero. Puedes operar con él directamente.
if edad >= 18:
    st.write("Mayor de edad")
```

Esto hace que el código se lea como un script normal, que es precisamente la intención del diseño.

### Consecuencia 2: el orden del código es el orden de la pantalla

Lo que escribes primero aparece arriba. No existe un fichero de layout aparte ni un sistema de plantillas. Si quieres que el título vaya sobre el gráfico, pon `st.title()` antes de `st.pyplot()`.

### Consecuencia 3: todo se recalcula constantemente

Y aquí está el gran peligro. Considera este código:

```python
import streamlit as st
import pandas as pd
import joblib

df = pd.read_csv("ventas_5_millones.csv")      # tarda 30 segundos
modelo = joblib.load("modelo_grande.joblib")    # tarda 15 segundos

umbral = st.slider("Umbral", 0.0, 1.0, 0.5)
st.write(f"Filtrando por {umbral}")
```

Cada vez que el usuario mueve el slider un milímetro, el script se re-ejecuta desde la línea 1. Es decir: se vuelven a leer cinco millones de filas y a cargar el modelo. **45 segundos de espera por cada movimiento del ratón.**

La aplicación funciona, técnicamente. Pero es inutilizable. Esto es lo que resuelve el sistema de caché del apartado 6, y es la diferencia entre una demo presentable y una que da vergüenza enseñar.

### Consecuencia 4: no hay memoria entre ejecuciones

Cada re-ejecución empieza de cero. Las variables normales se reinicializan. Esto rompe intuiciones:

```python
contador = 0
if st.button("Sumar uno"):
    contador += 1
st.write(contador)
```

Este código **nunca** mostrará 2. Al pulsar el botón, el script se re-ejecuta: `contador` vuelve a valer 0, el botón devuelve `True`, se suma 1, y se muestra 1. Al pulsarlo otra vez, exactamente lo mismo.

Para recordar cosas entre ejecuciones hace falta `st.session_state`, que veremos en el apartado 7.

## 0.4 Streamlit frente a Gradio

Si vienes del módulo de Gradio, la diferencia de filosofía es esta:

```python
# GRADIO: declaras componentes y luego conectas eventos
import gradio as gr

def doblar(x):
    return x * 2

with gr.Blocks() as demo:
    entrada = gr.Slider(0, 10)
    salida = gr.Textbox()
    entrada.change(fn=doblar, inputs=entrada, outputs=salida)

demo.launch()
```

```python
# STREAMLIT: script lineal
import streamlit as st

entrada = st.slider("Valor", 0, 10)
st.write(entrada * 2)
```

Streamlit resulta más natural a quien viene de escribir scripts de análisis de datos. Gradio resulta más natural a quien piensa en términos de "un modelo con entradas y salidas".

**Criterio de elección:** si el centro de tu aplicación es un modelo (entra un dato, sale una predicción), Gradio suele ser más directo. Si el centro son los datos (filtrar, explorar, visualizar), Streamlit es claramente mejor.

# Módulo 1. Ejecución en local

Este módulo cubre cómo trabajar en tu propio ordenador. Es el entorno recomendado para desarrollar, porque los cambios se ven al instante y no dependes de internet.


## 1.1 Requisitos previos

Necesitas **Python 3.9 o superior** instalado. Compruébalo abriendo una terminal:

```bash
python --version
```

Si el comando no existe, prueba con `python3 --version`. En Windows, si no funciona ninguno, instala Python desde [python.org](https://www.python.org/downloads/) marcando la casilla **"Add Python to PATH"** durante la instalación.

## 1.2 Crear un entorno virtual (recomendado)

Un entorno virtual es una carpeta que contiene una instalación de Python aislada, con sus propias librerías. Es la forma de evitar que las dependencias de un proyecto rompan las de otro.

No es obligatorio, pero te ahorrará problemas, sobre todo cuando llegues al despliegue y necesites saber exactamente qué librerías usa tu aplicación.

```bash
# 1. Crear una carpeta para el proyecto
mkdir mi-app-streamlit
cd mi-app-streamlit

# 2. Crear el entorno virtual (crea una subcarpeta llamada 'venv')
python -m venv venv

# 3. Activarlo
# En Linux o macOS:
source venv/bin/activate
# En Windows (PowerShell):
venv\Scripts\Activate.ps1
# En Windows (CMD):
venv\Scripts\activate.bat
```

Sabrás que está activado porque el prompt de la terminal cambia y aparece `(venv)` al principio:

```
(venv) usuario@ordenador:~/mi-app-streamlit$
```

Para desactivarlo, escribe `deactivate`.

> **Si usas Windows y PowerShell te da un error de permisos** al activar el entorno, ejecuta primero:
> ```powershell
> Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser
> ```


## 1.3 Instalar Streamlit

Con el entorno activado:

```python
pip install streamlit
```

Tarda un par de minutos porque arrastra bastantes dependencias. Comprueba que ha funcionado:

```bash
streamlit --version
```

Y para ver una demostración completa de lo que puede hacer:

```bash
streamlit hello
```

Esto abre el navegador con una aplicación de ejemplo. Ciérrala con **Ctrl+C** en la terminal.

## 1.4 Tu primera aplicación

Crea un fichero llamado `app.py` en la carpeta del proyecto con este contenido:

```python
import streamlit as st

st.title("Mi primera aplicación")

nombre = st.text_input("¿Cómo te llamas?")

if nombre:
    st.write(f"Hola, {nombre}. Bienvenido a Streamlit.")
else:
    st.info("Escribe tu nombre arriba para empezar.")
```

## 1.5 El comando de ejecución

Aquí está el error número uno de los principiantes. Una aplicación de Streamlit **no se ejecuta con `python`**:

```bash
python app.py            # MAL
streamlit run app.py     # BIEN
```

Si lo lanzas con `python`, no obtienes un error claro. Streamlit imprime un aviso por consola y el script termina sin hacer nada visible:

```
Warning: to view this Streamlit app on a browser, run it with the following
command:

    streamlit run app.py [ARGUMENTS]
```

Mucha gente pierde media hora aquí. Es fácil de recordar si piensas que `streamlit run` no ejecuta el script una vez: **arranca un servidor web que ejecutará el script cada vez que alguien interactúe**.

Al lanzarlo correctamente verás:

```
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.1.42:8501
```

Y se abrirá el navegador automáticamente.

**Qué significa cada URL:**

- **Local URL** (`localhost:8501`): funciona solo en tu ordenador. Es la que usarás mientras desarrollas.
- **Network URL** (una IP de tu red): funciona desde otros dispositivos conectados a tu misma red wifi. Útil para probar en el móvil, o para enseñársela a alguien en la misma oficina. **No funciona desde internet.**

## 1.6 El ciclo de desarrollo

Con la aplicación en marcha, edita `app.py` en tu editor y guarda. En el navegador aparecerá arriba a la derecha un mensaje:

> **Source file changed.** *Rerun* / *Always rerun*

Pulsa **Always rerun** una vez y a partir de ahí la aplicación se recargará sola cada vez que guardes. Este ciclo (editar, guardar, ver el resultado) es lo que hace Streamlit tan cómodo para prototipar.

Para detener el servidor, vuelve a la terminal y pulsa **Ctrl+C**.

## 1.7 Opciones útiles del comando

```bash
# Cambiar el puerto (útil si el 8501 está ocupado)
streamlit run app.py --server.port 8502

# No abrir el navegador automáticamente
streamlit run app.py --server.headless true

# Escuchar en todas las interfaces de red (necesario en Docker o servidores)
streamlit run app.py --server.address 0.0.0.0

# Desactivar el envío de estadísticas de uso
streamlit run app.py --browser.gatherUsageStats false
```

## 1.8 Problemas frecuentes en local

| Síntoma | Causa | Solución |
|---|---|---|
| `streamlit: command not found` | El entorno virtual no está activado, o la instalación falló | Actívalo, o usa `python -m streamlit run app.py` |
| `Port 8501 is already in use` | Hay otra instancia corriendo | Ciérrala con Ctrl+C, o usa `--server.port 8502` |
| El navegador no se abre solo | Configuración del sistema | Abre manualmente `http://localhost:8501` |
| Los cambios no se reflejan | No has pulsado *Rerun* | Pulsa **Always rerun** en el aviso |
| `ModuleNotFoundError` | Falta instalar una librería | `pip install nombre_libreria` |
| El aviso "to view this Streamlit app..." | Lo lanzaste con `python` | Usa `streamlit run` |

---

# Módulo 2. Ejecución desde Google Colab con ngrok

Este módulo es necesario si trabajas en Google Colab en lugar de en tu ordenador.

## 2.1 Por qué Colab necesita algo extra

Cuando ejecutas `streamlit run` en tu ordenador, el servidor web queda escuchando en `localhost:8501` y tu navegador puede acceder a él porque **está en la misma máquina**.

En Colab, la situación es distinta. Tu cuaderno se ejecuta en una máquina virtual en un centro de datos de Google, y tu navegador está en tu casa. Cuando Streamlit arranca en `localhost:8501`, ese `localhost` se refiere a **la máquina de Google**, no a la tuya. Y esa máquina no expone sus puertos a internet por seguridad.

El resultado es que el servidor arranca correctamente pero **no puedes verlo desde ningún sitio**.

```
Tu navegador  ──── internet ────  ✗  ──── Máquina virtual de Colab
                                            └── Streamlit en :8501
                                                (inaccesible desde fuera)
```

## 2.2 Qué es ngrok y qué hace exactamente

ngrok es un servicio de **túnel inverso**. Funciona así: un pequeño programa (el *agente*) se ejecuta en la máquina de Colab y **abre una conexión saliente** hacia los servidores de ngrok. Como es una conexión saliente, el cortafuegos la permite.

A partir de ese momento, ngrok te asigna una URL pública. Cuando alguien visita esa URL, la petición llega a los servidores de ngrok, que la reenvían por el túnel ya abierto hasta tu aplicación en Colab.

```
Tu navegador ──→ https://abc123.ngrok-free.app ──→ Servidores ngrok
                                                          │
                                                    (túnel abierto)
                                                          ↓
                                              Máquina de Colab :8501
```

`pyngrok` es simplemente una librería de Python que controla ese agente sin salir del cuaderno.

## 2.3 Obtener la cuenta y el token

ngrok exige registrarse y usar un token de autenticación. Es gratuito.

**Paso 1.** Entra en [ngrok.com](https://ngrok.com) y pulsa **Sign up**. Puedes registrarte con Google o GitHub, que es lo más rápido.

**Paso 2.** Una vez dentro, ve al panel de control. En el menú lateral izquierdo busca la sección **Getting Started** y dentro **Your Authtoken**. La URL directa es:

```
https://dashboard.ngrok.com/get-started/your-authtoken
```

**Paso 3.** Verás una caja con tu token, una cadena larga parecida a esta:

```
2abcDEFghiJKLmnoPQRstuVWXyz_1a2B3c4D5e6F7g8H9i
```

Pulsa el botón de copiar. Ese token identifica tu cuenta: **trátalo como una contraseña**, no lo publiques en GitHub ni lo compartas.

**Paso 4 (recomendado).** Guárdalo en los *Secrets* de Colab en lugar de escribirlo en una celda. En el panel izquierdo de Colab, pulsa el icono de la **llave 🔑**, crea un secreto llamado `NGROK_TOKEN` con el valor copiado, y activa el interruptor de acceso al cuaderno.

## 2.4 El procedimiento completo en Colab

Ahora las celdas, una a una y explicando qué hace cada una.

**Celda 1: instalar las librerías**


In [ ]:
!pip install -q streamlit pyngrok

La opción `-q` (*quiet*) reduce la cantidad de texto que imprime pip. Tarda alrededor de un minuto.

**Celda 2: escribir el fichero de la aplicación**

Aquí está el truco que conviene entender. Streamlit necesita un **fichero `.py`**, no celdas de cuaderno. La magia `%%writefile` de Jupyter convierte el contenido de una celda en un fichero:

In [ ]:
%%writefile app.py
import streamlit as st

st.title("Mi aplicación desde Colab")

nombre = st.text_input("¿Cómo te llamas?")
if nombre:
    st.write(f"Hola, {nombre}")

Al ejecutar la celda, Colab responde `Writing app.py` y el fichero queda creado en el sistema de ficheros de la máquina virtual. Puedes verlo en el panel de archivos (el icono de la carpeta, a la izquierda).

**Detalles importantes de `%%writefile`:**

- Debe ser **la primera línea de la celda**, sin nada antes, ni siquiera un comentario.
- El contenido de la celda **no se ejecuta**, solo se escribe al fichero. Por eso no verás errores de Python aunque los haya: los verás después, al arrancar Streamlit.
- Si vuelves a ejecutar la celda, el fichero se sobrescribe. Usa `%%writefile -a app.py` para añadir en lugar de sobrescribir.

**Celda 3: configurar el token de ngrok**

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Lee el token desde los Secrets de Colab (icono de la llave)
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
print("Token configurado")

Si prefieres no usar Secrets, la alternativa directa (menos segura) es:

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("PEGA_AQUI_TU_TOKEN")

**Celda 4: arrancar Streamlit y abrir el túnel**

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# 1. Cerrar túneles anteriores.
#    Sin esto, al reejecutar la celda acumularías túneles y la cuenta
#    gratuita solo permite uno simultáneo.
ngrok.kill()

# 2. Arrancar Streamlit en segundo plano.
#    --server.headless true evita que intente abrir un navegador
#    (no existe navegador en la máquina de Colab).
proceso = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--browser.gatherUsageStats", "false",
])

# 3. Dar tiempo a que el servidor termine de arrancar
time.sleep(5)

# 4. Abrir el túnel hacia el puerto 8501
url = ngrok.connect(8501)
print(f"Tu aplicación está en: {url}")

La salida será algo así:

```
Tu aplicación está en: NgrokTunnel: "https://a1b2-34-56-78-90.ngrok-free.app" -> "http://localhost:8501"
```

Haz clic en esa URL. La primera vez, ngrok muestra una **página intermedia de advertencia** con un botón **"Visit Site"**; púlsalo y llegarás a tu aplicación.

**Celda 5 (opcional): detener todo**


In [ ]:
ngrok.kill()
proceso.terminate()
print("Túnel cerrado y servidor detenido")

## 2.5 Ver los errores de tu aplicación

Este es el punto más incómodo de trabajar en Colab: **si tu aplicación tiene un error de Python, no lo verás en la salida de la celda**, porque Streamlit corre en segundo plano.

Hay dos formas de verlo:

**Opción A: el propio navegador.** Streamlit muestra los errores directamente en la página, con el traceback completo. Suele ser suficiente.

**Opción B: redirigir la salida a un fichero.** Modifica la celda 4 para capturar los logs:

In [ ]:
proceso = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501",
     "--server.headless", "true"],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)

Y luego, en otra celda:

In [ ]:
!cat streamlit.log

## 2.6 Limitaciones de ngrok con cuenta gratuita

Conviene conocerlas antes de depender de esto:

| Limitación | Detalle |
|---|---|
| **Un solo túnel simultáneo** | Si intentas abrir dos, el segundo falla. De ahí el `ngrok.kill()` inicial. |
| **URL aleatoria** | Cambia cada vez que reinicias el túnel, salvo que configures un dominio estático. |
| **Página de advertencia** | Los visitantes ven una pantalla intermedia antes de entrar. |
| **Depende de Colab** | Si se cierra la pestaña o expira la sesión, la URL muere. |
| **Límite de peticiones** | La cuenta gratuita limita el número de conexiones por minuto. |

La más importante es la última de la tabla: **esto no es un despliegue**. Sirve para probar y para enseñar algo en el momento, pero la URL desaparece en cuanto se cierra Colab. Para publicar de verdad, el apartado 9.

## 2.7 Dominio estático gratuito (opcional)

Desde 2023, ngrok ofrece un dominio estático gratuito por cuenta, lo que evita que la URL cambie en cada reinicio. Se configura así:

**Paso 1.** En el panel de ngrok, ve a **Domains** (o *Cloud Edge → Domains*) y crea uno. Te dará algo como `panda-new-kit.ngrok-free.app`.

**Paso 2.** Úsalo al conectar:

In [ ]:
url = ngrok.connect(8501, domain="panda-new-kit.ngrok-free.app")
print(f"URL fija: {url}")

Ahora la dirección será siempre la misma, lo cual es cómodo si vas a compartirla varias veces con las mismas personas.


## 2.8 Alternativas a ngrok

ngrok es la opción más documentada, pero no la única.

**localtunnel**

No requiere registro ni token:


In [ ]:
!npm install -g localtunnel
!streamlit run app.py --server.port 8501 &>/dev/null &
!npx localtunnel --port 8501

Te pedirá una contraseña, que es la IP pública de la máquina de Colab. La obtienes con:


In [ ]:
!curl https://loca.lt/mytunnelpassword

Es más simple de arrancar, pero menos estable.

**El método nativo de Colab**

Colab tiene una función propia para exponer puertos, sin servicios externos:

In [ ]:
from google.colab import output
output.serve_kernel_port_as_website(8501)

Genera un enlace que **solo funciona para ti**, en tu sesión de Colab. No sirve para compartir, pero es perfecto para desarrollar sin depender de ngrok.

**Cuál elegir**

| Necesidad | Herramienta |
|---|---|
| Solo quiero ver mi app mientras desarrollo en Colab | `output.serve_kernel_port_as_website` |
| Quiero enseñársela a alguien ahora mismo | ngrok |
| No quiero registrarme en nada | localtunnel |
| Quiero una URL permanente | Ninguna de estas: ve al apartado 9 |

---

# Módulo 3. Widgets de entrada

Todos los widgets funcionan igual: se llaman, dibujan el control **y devuelven el valor actual**. Este módulo los recorre uno a uno con sus parámetros importantes.

## 3.1 Parámetros comunes

Casi todos los widgets aceptan estos argumentos, y conviene conocerlos antes de ver los widgets concretos:

```python
st.slider(
    label="Edad",           # el texto visible sobre el control
    ...                     # parámetros específicos del widget
    help="Edad en años",    # icono de interrogación con explicación
    key="edad_usuario",     # identificador único (ver 3.10)
    disabled=False,         # si True, se ve pero no se puede usar
    label_visibility="visible",   # "hidden" oculta la etiqueta, "collapsed" la elimina
    on_change=funcion,      # callback opcional (ver módulo 7)
)
```

El parámetro `help` es el más infravalorado: añade un pequeño icono de interrogación que muestra una explicación al pasar el ratón. Es la forma más barata de hacer una interfaz comprensible sin llenarla de texto.

## 3.2 Entrada de texto

### `st.text_input()`

```python
nombre = st.text_input(
    "Nombre completo",
    value="",                       # valor inicial
    max_chars=50,                   # límite de caracteres
    placeholder="Ej: Ana García",   # texto gris cuando está vacío
    type="default",                 # "password" oculta lo escrito
)
```

Devuelve un `str`. Si el usuario no ha escrito nada, devuelve la cadena vacía `""`, que en Python es *falsy*. Por eso el patrón habitual es:

```python
if nombre:
    st.write(f"Hola, {nombre}")
```

El tipo `password` es útil para claves de API:

```python
api_key = st.text_input("Clave de API", type="password")
```

### `st.text_area()`

Para texto de varias líneas:

```python
comentario = st.text_area(
    "Describe el problema",
    height=150,          # altura en píxeles
    max_chars=1000,
)
```

## 3.3 Entrada numérica


### `st.number_input()`

Una caja con flechas de incremento. Útil cuando el usuario debe introducir un valor **preciso**:

```python
precio = st.number_input(
    "Precio",
    min_value=0.0,
    max_value=10000.0,
    value=99.99,          # valor inicial
    step=0.01,            # incremento de las flechas
    format="%.2f",        # formato de visualización
)
```

**Detalle importante sobre el tipo devuelto:** Streamlit decide si devuelve `int` o `float` según el tipo de los argumentos. Si escribes `min_value=0` devuelve enteros; si escribes `min_value=0.0` devuelve decimales. Mezclar tipos (`min_value=0, value=1.5`) da error.


### `st.slider()`

Un deslizador. Mejor que `number_input` cuando el rango está acotado y el valor exacto importa menos que la exploración:

```python
temperatura = st.slider(
    "Temperatura",
    min_value=0.0,
    max_value=2.0,
    value=0.7,
    step=0.1,
)
```

Admite también **rangos**, pasando una tupla como valor inicial. En ese caso devuelve una tupla:

```python
rango = st.slider("Rango de precios", 0, 1000, (200, 800))
minimo, maximo = rango
st.write(f"Filtrando entre {minimo} y {maximo}")
```

Y funciona con fechas:

```python
import datetime
fechas = st.slider(
    "Periodo",
    min_value=datetime.date(2024, 1, 1),
    max_value=datetime.date(2026, 12, 31),
    value=(datetime.date(2025, 1, 1), datetime.date(2025, 12, 31)),
)
```

### `st.select_slider()`

Un deslizador sobre valores **no numéricos**, respetando su orden:

```python
nivel = st.select_slider(
    "Nivel de riesgo",
    options=["Muy bajo", "Bajo", "Medio", "Alto", "Muy alto"],
    value="Medio",
)
```

Perfecto para escalas ordinales, donde un slider numérico no tendría sentido.

## 3.4 Selección entre opciones


### `st.selectbox()`

Un desplegable. La opción por defecto cuando hay más de cuatro o cinco alternativas:

```python
algoritmo = st.selectbox(
    "Algoritmo",
    options=["Random Forest", "Regresión Logística", "SVM", "XGBoost"],
    index=0,               # cuál viene seleccionada (0 = la primera)
    help="Random Forest suele funcionar bien sin ajustar nada",
)
```

Devuelve **el elemento seleccionado**, no su índice.

Truco útil: si quieres permitir "ninguna selección", pon `index=None` y añade un `placeholder`:

```python
columna = st.selectbox(
    "Columna objetivo",
    options=list(df.columns),
    index=None,
    placeholder="Selecciona una columna...",
)
if columna is None:
    st.info("Elige una columna para continuar")
    st.stop()
```

### `st.radio()`

Botones de opción. Preferible al desplegable cuando hay **pocas opciones** y quieres que todas sean visibles a la vez:

```python
modo = st.radio(
    "Modo de análisis",
    options=["Rápido", "Completo"],
    horizontal=True,       # los coloca en fila en lugar de en columna
    captions=["~5 segundos", "~2 minutos"],   # texto pequeño bajo cada opción
)
```

### `st.multiselect()`

Selección múltiple. Devuelve una **lista**:

```python
variables = st.multiselect(
    "Variables a incluir en el modelo",
    options=list(df.columns),
    default=list(df.columns[:3]),     # preseleccionadas
)

if not variables:
    st.warning("Selecciona al menos una variable")
    st.stop()

st.write(f"Has elegido {len(variables)} variables")
```

### `st.checkbox()`

Devuelve `True` o `False`:

```python
normalizar = st.checkbox("Normalizar los datos", value=True)

if normalizar:
    X = scaler.fit_transform(X)
```

Un uso muy práctico es mostrar u ocultar secciones enteras:

```python
if st.checkbox("Mostrar datos en bruto"):
    st.dataframe(df)
```

## 3.5 Fechas y horas


### `st.date_input`

```python
import datetime

fecha = st.date_input(
    "Fecha de análisis",
    value=datetime.date.today(),
    min_value=datetime.date(2020, 1, 1),
    max_value=datetime.date.today(),
)

hora = st.time_input("Hora de inicio", value=datetime.time(9, 0))
```

`st.date_input` también admite rangos pasando una tupla, y en ese caso devuelve una tupla de dos fechas.

## 3.6 Subida de ficheros


### `st.file_uploader()`

```python
archivo = st.file_uploader(
    "Sube tu fichero de datos",
    type=["csv", "xlsx"],        # extensiones permitidas
    accept_multiple_files=False,
    help="Máximo 200 MB",
)
```

Devuelve `None` si no hay fichero, o un objeto que **se comporta como un fichero abierto**. Esto es una diferencia notable con Gradio:

```python
# En Gradio había que sacar la ruta:
df = pd.read_csv(archivo.name)

# En Streamlit se pasa directamente:
df = pd.read_csv(archivo)
```

El patrón completo y robusto:

```python
archivo = st.file_uploader("Sube un CSV", type=["csv"])

if archivo is None:
    st.info("Sube un fichero para empezar el análisis.")
    st.stop()

try:
    df = pd.read_csv(archivo)
except Exception as e:
    st.error(f"No se ha podido leer el fichero: {e}")
    st.stop()

st.success(f"Fichero cargado: {len(df)} filas y {len(df.columns)} columnas")
st.dataframe(df.head())
```

Con `accept_multiple_files=True` devuelve una lista de objetos, y hay que iterar:

```python
archivos = st.file_uploader("Sube varios CSV", type=["csv"], accept_multiple_files=True)
for archivo in archivos:
    df = pd.read_csv(archivo)
    st.write(f"{archivo.name}: {len(df)} filas")
```

Fíjate en que aquí sí se usa `.name`, pero solo para **mostrar** el nombre, no para leerlo.

### `st.camera_input()`

Toma una foto con la webcam. Devuelve un objeto de imagen o `None`:

```python
foto = st.camera_input("Haz una foto")
if foto is not None:
    from PIL import Image
    imagen = Image.open(foto)
    st.image(imagen)
```

## 3.7 Botones


### `st.button()`

```python
if st.button("Entrenar modelo", type="primary"):
    entrenar()
```

El parámetro `type="primary"` lo destaca visualmente. `type="secondary"` es el estilo normal.

**Aquí está el comportamiento más confuso de todo Streamlit**, y merece explicación detallada.

`st.button()` devuelve `True` **únicamente durante la re-ejecución que provoca el propio clic**. En la siguiente interacción vuelve a `False`.

Veámoslo con un caso que falla:

```python
if st.button("Entrenar"):
    modelo = entrenar()        # se entrena
    st.success("Entrenado")

umbral = st.slider("Umbral", 0.0, 1.0, 0.5)
st.write(modelo.predict(...))   # ERROR: 'modelo' no existe
```

Al pulsar "Entrenar", el script se re-ejecuta, el botón devuelve `True`, se entrena y se muestra el mensaje. Pero en cuanto el usuario mueve el slider, el script se re-ejecuta **otra vez**, el botón ya devuelve `False`, el bloque no se ejecuta, y la variable `modelo` no llega a crearse. El programa falla con `NameError`.

La solución es `st.session_state` (módulo 7):

```python
if st.button("Entrenar"):
    st.session_state.modelo = entrenar()

if "modelo" in st.session_state:
    umbral = st.slider("Umbral", 0.0, 1.0, 0.5)
    st.write(st.session_state.modelo.predict(...))
```


### `st.download_button()`

Permite descargar un fichero generado por la aplicación:

```python
csv = df.to_csv(index=False).encode("utf-8")

st.download_button(
    label="Descargar resultados en CSV",
    data=csv,
    file_name="resultados.csv",
    mime="text/csv",
)
```

El argumento `data` acepta `str`, `bytes` o un objeto tipo fichero. Para Excel:

```python
import io
buffer = io.BytesIO()
df.to_excel(buffer, index=False)
st.download_button(
    "Descargar Excel",
    data=buffer.getvalue(),
    file_name="resultados.xlsx",
    mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
)
```

## 3.8 Otros widgets


```python
color = st.color_picker("Color del gráfico", value="#2563eb")

# Editor de tablas: el usuario puede modificar celdas
df_editado = st.data_editor(df, num_rows="dynamic")

# Entrada de chat (para asistentes conversacionales)
mensaje = st.chat_input("Escribe tu pregunta...")
```

`st.data_editor` es especialmente potente: devuelve el DataFrame **con las modificaciones del usuario**, lo que permite construir interfaces de corrección de datos con muy poco código.

## 3.9 Widgets en la barra lateral

Cualquier widget puede colocarse en la barra lateral de dos formas equivalentes:

```python
# Forma 1: con un bloque with (recomendada si hay varios)
with st.sidebar:
    st.header("Configuración")
    modelo = st.selectbox("Algoritmo", ["RF", "SVM"])
    n_arboles = st.slider("Nº de árboles", 10, 500, 100)

# Forma 2: prefijo directo (cómoda para uno suelto)
umbral = st.sidebar.slider("Umbral", 0.0, 1.0, 0.5)
```

**Convención muy recomendable:** los controles de configuración van en la barra lateral y los resultados en el área principal. Es lo que los usuarios esperan y mantiene la pantalla despejada.

## 3.10 La clave `key` y por qué la necesitas

Streamlit identifica cada widget por una combinación de su tipo, su etiqueta y sus parámetros. Si creas dos widgets idénticos, no puede distinguirlos y lanza el error `DuplicateWidgetID`.

Esto ocurre típicamente dentro de bucles:

```python
# ESTO FALLA
for columna in df.columns:
    st.slider("Peso", 0.0, 1.0, 0.5)      # DuplicateWidgetID

# ESTO FUNCIONA
for i, columna in enumerate(df.columns):
    st.slider(f"Peso de {columna}", 0.0, 1.0, 0.5, key=f"peso_{i}")
```

La `key` tiene además un segundo uso muy importante: **el valor del widget queda accesible en `st.session_state`**:

```python
st.slider("Umbral", 0.0, 1.0, 0.5, key="umbral")

# En cualquier punto posterior del script:
st.write(st.session_state.umbral)
```

Y esto permite modificar un widget desde código, cosa que veremos en el módulo 7.

---

# Módulo 4. Elementos de salida

## 4.1 Texto y estructura

```python
st.title("Título de la aplicación")        # el más grande
st.header("Sección principal")
st.subheader("Subsección")
st.markdown("Texto con **negrita**, *cursiva*, `código` y [enlaces](https://x.com)")
st.caption("Texto pequeño y gris, para notas y aclaraciones")
st.text("Texto monoespaciado sin formato")
st.divider()                                # línea horizontal separadora
```

`st.markdown` acepta Markdown completo, incluidas listas, tablas y citas. También admite HTML si añades `unsafe_allow_html=True`, aunque conviene evitarlo salvo necesidad real.

Para código y matemáticas:

```python
st.code("""
def predecir(x):
    return modelo.predict([x])[0]
""", language="python")

st.latex(r"\text{accuracy} = \frac{VP + VN}{VP + VN + FP + FN}")
```

### `st.write()`: el comodín

`st.write()` detecta el tipo de lo que le pasas y elige la representación adecuada:

```python
st.write("Un texto")             # → markdown
st.write(df)                     # → tabla interactiva
st.write(fig)                    # → gráfico
st.write({"a": 1, "b": 2})       # → JSON desplegable
st.write(modelo)                 # → representación del objeto
st.write("Texto", 42, df)        # → varios elementos seguidos
```

Es muy cómodo para explorar y depurar. En código definitivo, sin embargo, es mejor usar la función específica (`st.dataframe`, `st.pyplot`...) porque permite configurar la presentación.


## 4.2 Mostrar datos


### `st.dataframe()` frente a `st.table()`

```python
st.dataframe(df)      # interactiva: ordenable, con scroll, buscable
st.table(df)          # estática: se pinta entera, sin interacción
```

Para resultados de modelos y exploración de datos, `st.dataframe` casi siempre es la elección correcta. `st.table` sirve para tablas pequeñas que quieres que se vean completas sin scroll.

Parámetros útiles de `st.dataframe`:

```python
st.dataframe(
    df,
    width="stretch",        # ocupa todo el ancho disponible
    height=400,             # altura fija en píxeles
    hide_index=True,        # oculta la columna de índice
    column_config={         # configuración por columna
        "precio": st.column_config.NumberColumn("Precio", format="%.2f €"),
        "url": st.column_config.LinkColumn("Enlace"),
        "progreso": st.column_config.ProgressColumn("Avance", min_value=0, max_value=1),
    },
)
```

> **Aviso de versión:** el parámetro `use_container_width=True` que verás en tutoriales anteriores a 2026 está **obsoleto y ya retirado**. Ahora se usa `width="stretch"` para ocupar todo el ancho, o `width="content"` para ajustar al contenido.

### `st.metric()`

Muestra un número destacado, opcionalmente con su variación:

```python
st.metric(
    label="Accuracy",
    value="96.7%",
    delta="+2.1%",              # flecha verde hacia arriba
    delta_color="normal",       # "inverse" invierte los colores
    help="Medida en validación cruzada con 5 particiones",
)
```

Combinado con columnas, es la forma habitual de presentar resultados:

```python
c1, c2, c3 = st.columns(3)
c1.metric("Precisión", "0.94")
c2.metric("Exhaustividad", "0.91")
c3.metric("F1", "0.92", delta="-0.01")
```

El parámetro `delta_color="inverse"` es útil cuando un aumento es malo (por ejemplo, tiempo de ejecución o tasa de error).

### `st.json()`

```python
st.json({"clase": "setosa", "probabilidad": 0.982, "modelo": "RF-100"})
```

Muestra un visor plegable, útil para respuestas de API o configuraciones.

## 4.3 Gráficos


### Gráficos nativos

Streamlit incluye funciones de gráfico muy rápidas de usar, pensadas para explorar:

```python
st.line_chart(df, x="fecha", y="ventas")
st.bar_chart(df, x="categoria", y="total")
st.area_chart(df)
st.scatter_chart(df, x="altura", y="peso", color="grupo", size="edad")
st.map(df)      # necesita columnas 'lat' y 'lon'
```

Son cómodos pero poco configurables. Sirven para el 80 % de los casos exploratorios.


### matplotlib

```python
from matplotlib.figure import Figure

fig = Figure(figsize=(8, 4))
ax = fig.subplots()
ax.bar(nombres, valores, color="#2563eb")
ax.set_ylabel("Probabilidad")
ax.set_title("Distribución")
fig.tight_layout()

st.pyplot(fig)
```

**Detalle técnico importante.** Verás en muchos tutoriales esta forma:

```python
import matplotlib.pyplot as plt
fig, ax = plt.subplots()      # NO recomendado en Streamlit
```

El problema es que `pyplot` mantiene un **registro global de todas las figuras creadas**. Como el script de Streamlit se re-ejecuta constantemente, cada interacción del usuario genera una figura nueva que nunca se libera. En una sesión larga se acumulan cientos y la aplicación acaba consumiendo toda la memoria disponible.

Usando la clase `Figure` directamente, las figuras son objetos normales que el recolector de basura de Python libera solo. Es un cambio de dos líneas que evita un fallo difícil de diagnosticar.

Si prefieres seguir con `pyplot`, cierra la figura explícitamente:

```python
st.pyplot(fig)
plt.close(fig)
```

### Plotly y Altair

Para gráficos interactivos (zoom, tooltips, selección):

```python
import plotly.express as px

fig = px.scatter(df, x="altura", y="peso", color="grupo",
                 hover_data=["nombre"])
st.plotly_chart(fig, width="stretch")
```


## 4.4 Mensajes al usuario

```python
st.success("Modelo entrenado correctamente")     # verde
st.info("El conjunto tiene 150 filas")           # azul
st.warning("La confianza es baja, revisa los datos")   # naranja
st.error("La columna 'target' no existe")        # rojo
st.exception(e)                                   # traceback formateado
```

Y un extra visual:

```python
st.balloons()     # globos subiendo (celebración)
st.snow()         # nieve
```

Úsalos con moderación; en una aplicación profesional pueden resultar fuera de lugar.

## 4.5 Indicadores de progreso


### Spinner: para operaciones cortas

```python
with st.spinner("Entrenando el modelo..."):
    modelo.fit(X, y)
st.success("Listo")
```

Muestra un indicador giratorio mientras dura el bloque. Es la forma más simple de evitar que el usuario piense que la aplicación se ha colgado.

### Barra de progreso: cuando conoces el avance

```python
barra = st.progress(0, text="Iniciando...")

for i in range(100):
    procesar_elemento(i)
    barra.progress((i + 1) / 100, text=f"Procesando {i+1} de 100")

barra.empty()      # elimina la barra al terminar
```

El valor debe estar entre 0.0 y 1.0 (o entre 0 y 100 si usas enteros).

### Status: para procesos de varias fases

```python
with st.status("Ejecutando pipeline...", expanded=True) as estado:
    st.write("Cargando datos...")
    df = cargar()

    st.write("Preprocesando...")
    X, y = preparar(df)

    st.write("Entrenando...")
    modelo = entrenar(X, y)

    estado.update(label="Pipeline completado", state="complete", expanded=False)
```

Muestra un panel plegable con el detalle de cada paso. Es lo más adecuado para procesos largos donde quieres que el usuario vea qué está pasando.

## 4.6 Multimedia

```python
st.image("grafico.png", caption="Distribución de clases", width=600)
st.image(array_numpy)                    # también acepta arrays y objetos PIL
st.audio("audio.mp3")
st.video("video.mp4")
```

---

# Módulo 5. Organización de la pantalla

## 5.1 Barra lateral

Ya vista en 3.9. La regla: **configuración a la izquierda, resultados en el centro**.

```python
with st.sidebar:
    st.header("Parámetros")
    st.divider()
    # ... controles ...
    st.divider()
    st.caption("Versión 1.2 · Julio 2026")
```

## 5.2 Columnas

```python
col1, col2 = st.columns(2)          # dos columnas iguales
```

El argumento puede ser un número (columnas iguales) o una lista de pesos:

```python
col1, col2 = st.columns([3, 1])     # la primera ocupa el 75 %
col1, col2, col3 = st.columns([2, 3, 2])
```

Hay dos formas de escribir dentro de una columna:

```python
# Forma 1: bloque with (mejor cuando hay varios elementos)
with col1:
    st.subheader("Gráfico")
    st.pyplot(fig)
    st.caption("Fuente: datos internos")

# Forma 2: método directo (cómoda para un solo elemento)
col2.metric("Total", "1.234")
```

Parámetros adicionales:

```python
col1, col2 = st.columns(2, gap="large", vertical_alignment="center")
```

`gap` acepta `"small"`, `"medium"` o `"large"`. `vertical_alignment` permite `"top"`, `"center"` o `"bottom"`.

**Las columnas se pueden anidar**, aunque conviene no abusar:

```python
izq, der = st.columns(2)
with izq:
    a, b = st.columns(2)
    a.metric("X", 1)
    b.metric("Y", 2)
```

## 5.3 Pestañas

```python
tab1, tab2, tab3 = st.tabs(["Predicción", "Datos", "Sobre el modelo"])

with tab1:
    st.subheader("Haz una predicción")
    # ...

with tab2:
    st.dataframe(df)

with tab3:
    st.markdown("Este modelo se entrenó con 150 muestras...")
```

Un matiz sobre rendimiento: **el contenido de todas las pestañas se ejecuta**, aunque no estén visibles. Si una pestaña contiene un cálculo pesado, se ejecutará igualmente. Para evitarlo, combina con `st.session_state` o con caché.

## 5.4 Expander: contenido plegable

```python
with st.expander("Opciones avanzadas", expanded=False):
    semilla = st.number_input("Semilla aleatoria", value=42)
    profundidad = st.slider("Profundidad máxima", 1, 20, 10)
```

Patrón muy recomendable: la interfaz principal queda limpia con lo esencial, y los parámetros técnicos están disponibles para quien los busque.

También sirve para documentación contextual:

```python
with st.expander("¿Cómo interpreto estos resultados?"):
    st.markdown("""
    La probabilidad indica la confianza del modelo, no la certeza.
    Un valor de 0.95 significa que...
    """)
```

## 5.5 Contenedores y huecos reservados


### `st.container()`

Agrupa elementos, opcionalmente con un borde:

```python
with st.container(border=True):
    st.subheader("Resumen")
    st.metric("Total", "1.234")
```

Tiene un uso menos evidente pero muy útil: **reservar un sitio en la página para rellenarlo después**. Como el script se ejecuta de arriba abajo, normalmente no puedes escribir arriba algo que calculas abajo. Con un contenedor sí:

```python
resumen = st.container()      # reserva el sitio aquí arriba

df = cargar_datos()
resultado = analizar(df)      # cálculo largo

with resumen:                 # pero se pinta arriba
    st.metric("Filas procesadas", len(df))
```

### `st.empty()`

Reserva un hueco cuyo contenido se **sustituye** en lugar de acumularse:

```python
hueco = st.empty()

for i in range(10):
    hueco.write(f"Procesando elemento {i}...")
    time.sleep(0.5)

hueco.success("Proceso completado")
```

Sin `st.empty()`, el bucle escribiría diez líneas. Con él, cada mensaje reemplaza al anterior.

## 5.6 Aplicaciones de varias páginas

Cuando la aplicación crece, conviene dividirla. Streamlit lo hace automáticamente si creas una carpeta `pages/`:

```
mi_app/
├── streamlit_app.py          # página principal
└── pages/
    ├── 1_📊_Exploración.py
    ├── 2_🤖_Modelo.py
    └── 3_🔮_Predicción.py
```

Streamlit genera la navegación en la barra lateral automáticamente. Reglas de nomenclatura:

- El **número inicial** fija el orden y no se muestra.
- Los **guiones bajos** se convierten en espacios.
- Los **emojis** se muestran como icono.

Así, `1_📊_Exploración.py` aparece en el menú como "📊 Exploración".

Cada página es un script independiente, pero **`st.session_state` se comparte entre todas**, lo que permite pasar datos de una página a otra.

---

# Módulo 6. Caché: el concepto crítico

Este es el módulo más importante del curso. Sin caché, una aplicación de Streamlit con un modelo real es inutilizable.

## 6.1 Recordatorio del problema

El script se re-ejecuta entero en cada interacción. Todo lo que esté en el cuerpo del script se vuelve a ejecutar: lecturas de disco, consultas a bases de datos, cargas de modelos, entrenamientos.

```python
import streamlit as st
import pandas as pd
import joblib

df = pd.read_csv("ventas.csv")                  # 30 s
modelo = joblib.load("modelo.joblib")            # 15 s

umbral = st.slider("Umbral", 0.0, 1.0, 0.5)
st.write(df[df["score"] > umbral])
```

Mover el slider cuesta 45 segundos. Cada vez.

## 6.2 La solución: mover el trabajo caro a una función decorada

```python
@st.cache_data
def cargar_datos():
    return pd.read_csv("ventas.csv")

@st.cache_resource
def cargar_modelo():
    return joblib.load("modelo.joblib")

df = cargar_datos()          # tarda 30 s la PRIMERA vez, 0 s las siguientes
modelo = cargar_modelo()     # tarda 15 s la PRIMERA vez, 0 s las siguientes

umbral = st.slider("Umbral", 0.0, 1.0, 0.5)
st.write(df[df["score"] > umbral])
```

Streamlit ejecuta la función la primera vez, guarda el resultado, y en las re-ejecuciones siguientes devuelve el valor almacenado sin ejecutar el cuerpo.

Dos requisitos para que esto funcione:

1. **El trabajo caro debe estar dentro de una función.** Si lo dejas en el cuerpo del script, no hay nada que cachear.
2. **La función debe estar decorada.**

## 6.3 `cache_data` frente a `cache_resource`

Elegir mal provoca errores sutiles, así que merece la pena entender la diferencia.

### `@st.cache_data`

Para **datos**: DataFrames, listas, diccionarios, arrays, resultados de consultas.

Streamlit **serializa** el resultado y devuelve una **copia nueva** a cada usuario y a cada ejecución. Eso significa que si tu código modifica el objeto devuelto, no afecta a lo que hay en caché ni a otros usuarios.

```python
@st.cache_data
def cargar_datos(ruta):
    return pd.read_csv(ruta)

df = cargar_datos("datos.csv")
df["nueva_columna"] = 1        # seguro: modifica solo tu copia
```

### `@st.cache_resource`

Para **recursos**: modelos de machine learning, conexiones a bases de datos, clientes de API, tokenizadores.

Streamlit guarda **el objeto en sí** y devuelve **la misma instancia** a todos los usuarios. No lo copia, porque copiar un modelo de 2 GB en cada sesión sería absurdo.

```python
@st.cache_resource
def cargar_modelo():
    return joblib.load("modelo.joblib")

modelo = cargar_modelo()       # todos los usuarios comparten este objeto
```

### La tabla de decisión

| | `@st.cache_data` | `@st.cache_resource` |
|---|---|---|
| Qué guarda | Una copia serializada | El objeto original |
| Qué devuelve | Copia nueva cada vez | Siempre la misma instancia |
| Requisito | El resultado debe ser serializable | Ninguno |
| Usar para | DataFrames, listas, dicts, arrays | Modelos, conexiones, clientes |
| Riesgo si te equivocas | Copiar un modelo enorme en cada sesión | Que dos usuarios se pisen los datos |

**Regla práctica:** si el objeto representa *datos que el código podría modificar*, usa `cache_data`. Si es un *recurso pesado que nadie va a modificar*, usa `cache_resource`.

Un modelo de scikit-learn entra siempre en `cache_resource`.

## 6.4 Cómo decide Streamlit si reutilizar el resultado

La caché se indexa por **el nombre de la función y sus argumentos**. Estas dos llamadas generan dos entradas distintas:

```python
cargar_datos("enero.csv")     # entrada 1
cargar_datos("febrero.csv")   # entrada 2
```

Y esto tiene una consecuencia que causa errores difíciles de encontrar: **si la función depende de algo que no es un argumento, la caché no se enterará de que ha cambiado**.

```python
# MAL: la caché no sabe que RUTA puede cambiar
RUTA = "datos.csv"

@st.cache_data
def cargar():
    return pd.read_csv(RUTA)     # depende de una global

# BIEN: la dependencia es explícita
@st.cache_data
def cargar(ruta):
    return pd.read_csv(ruta)

df = cargar(RUTA)
```

Streamlit también invalida la caché automáticamente si **cambias el código de la función**, lo cual es muy cómodo durante el desarrollo.

## 6.5 Parámetros del decorador

```python
@st.cache_data(ttl=3600)                    # caduca a los 3600 segundos
def consultar_api(endpoint):
    return requests.get(endpoint).json()

@st.cache_data(max_entries=10)              # guarda como mucho 10 resultados
def procesar(parametro):
    ...

@st.cache_data(show_spinner="Cargando datos...")   # mensaje personalizado
def cargar():
    ...

@st.cache_data(persist="disk")              # sobrevive al reinicio del servidor
def cargar_grande():
    ...
```

`ttl` (*time to live*) es imprescindible para datos que cambian: una consulta a una API o a una base de datos no debería cachearse indefinidamente.

`max_entries` evita que la caché crezca sin límite cuando la función se llama con muchos argumentos distintos.


## 6.6 Limpiar la caché

```python
cargar_datos.clear()        # limpia solo esta función
st.cache_data.clear()       # limpia toda la caché de datos
st.cache_resource.clear()   # limpia toda la caché de recursos
```

Un patrón útil para el usuario final:

```python
if st.sidebar.button("Recargar datos"):
    cargar_datos.clear()
    st.rerun()
```

## 6.7 Argumentos que no se pueden hashear

Para decidir si reutiliza un resultado, Streamlit calcula un *hash* de los argumentos. Algunos objetos no se pueden hashear y provocan un error `UnhashableParamError`.

La solución es prefijar el nombre del parámetro con un guion bajo, lo que indica a Streamlit que lo ignore al calcular la clave:

```python
@st.cache_data
def predecir_lote(_modelo, datos):
    return _modelo.predict(datos)
```

**Cuidado con el efecto secundario:** al ignorarse, si cambias el modelo la caché **no** se invalida y seguirás obteniendo resultados del modelo antiguo. Si eso puede ocurrir, añade un argumento que sí identifique la versión:

```python
@st.cache_data
def predecir_lote(_modelo, datos, version_modelo):
    return _modelo.predict(datos)

resultado = predecir_lote(modelo, X, version_modelo="v2.1")
```

## 6.8 Qué cachear y qué no

**Cachea:**
- Carga de ficheros y modelos
- Consultas a bases de datos y APIs
- Cálculos que tarden más de un segundo
- Preprocesamiento de datos

**No caches:**
- Funciones que devuelven algo distinto cada vez (números aleatorios, fecha actual)
- Funciones con efectos secundarios (escribir en disco, enviar correos)
- Operaciones triviales: el coste de gestionar la caché sería mayor que el cálculo

---

# Módulo 7. Estado, formularios y flujo de control

## 7.1 El problema del estado

Como cada re-ejecución empieza de cero, las variables normales no sobreviven:

```python
contador = 0
if st.button("Sumar"):
    contador += 1
st.write(contador)      # siempre 1, nunca 2
```

## 7.2 `st.session_state`

Es un diccionario que persiste entre re-ejecuciones, **individual para cada usuario** y para cada pestaña del navegador.

```python
# 1. Inicializar (solo la primera vez)
if "contador" not in st.session_state:
    st.session_state.contador = 0

# 2. Modificar
if st.button("Sumar"):
    st.session_state.contador += 1

# 3. Leer
st.write(st.session_state.contador)      # ahora sí: 1, 2, 3...
```

El paso 1 es obligatorio: sin la comprobación `if "clave" not in ...`, cada re-ejecución reiniciaría el valor a cero.

Se accede de dos formas equivalentes:

```python
st.session_state.contador
st.session_state["contador"]
```

La segunda es necesaria si la clave tiene espacios o caracteres especiales.

Y hay un método `get` como en cualquier diccionario, útil para valores opcionales:

```python
modelo = st.session_state.get("modelo", None)
if modelo is None:
    st.info("Entrena un modelo primero")
    st.stop()
```

## 7.3 Casos de uso reales


### Guardar un modelo entrenado por el usuario

```python
if st.button("Entrenar modelo"):
    with st.spinner("Entrenando..."):
        st.session_state.modelo = entrenar(X, y)
        st.session_state.metricas = evaluar(st.session_state.modelo, X, y)
    st.success("Modelo entrenado")

if "modelo" in st.session_state:
    st.metric("Accuracy", f"{st.session_state.metricas['acc']:.3f}")

    umbral = st.slider("Umbral", 0.0, 1.0, 0.5)
    st.write(st.session_state.modelo.predict_proba(X_nuevo))
```

Sin `session_state`, el modelo se perdería en cuanto el usuario tocase el slider.


### Historial de predicciones

```python
if "historial" not in st.session_state:
    st.session_state.historial = []

entrada = st.number_input("Valor")
if st.button("Predecir"):
    resultado = modelo.predict([[entrada]])[0]
    st.session_state.historial.append({"entrada": entrada, "salida": resultado})

if st.session_state.historial:
    st.dataframe(pd.DataFrame(st.session_state.historial))
    if st.button("Limpiar historial"):
        st.session_state.historial = []
        st.rerun()
```

### Asistentes de varios pasos

```python
if "paso" not in st.session_state:
    st.session_state.paso = 1

if st.session_state.paso == 1:
    st.header("Paso 1: sube los datos")
    archivo = st.file_uploader("CSV")
    if archivo and st.button("Siguiente"):
        st.session_state.datos = pd.read_csv(archivo)
        st.session_state.paso = 2
        st.rerun()

elif st.session_state.paso == 2:
    st.header("Paso 2: elige las variables")
    # ...
```

## 7.4 Widgets y `session_state`

Un widget con `key` guarda automáticamente su valor en `session_state`:

```python
st.slider("Umbral", 0.0, 1.0, 0.5, key="umbral")
st.write(st.session_state.umbral)      # el mismo valor
```

Esto permite **modificar un widget desde código**:

```python
if st.button("Restablecer valores"):
    st.session_state.umbral = 0.5
    st.session_state.n_arboles = 100
    st.rerun()
```

> **Advertencia importante:** no puedes modificar la clave de un widget **después** de haberlo creado en la misma ejecución. Streamlit lanza `StreamlitAPIException`. Hazlo antes de la línea que crea el widget, o usa `st.rerun()`.

## 7.5 Formularios

Un formulario agrupa varios widgets y **retrasa la re-ejecución hasta que se pulsa el botón de envío**.

```python
with st.form("formulario_paciente"):
    st.subheader("Datos del paciente")

    edad = st.number_input("Edad", 0, 120, 45)
    peso = st.number_input("Peso (kg)", 0.0, 300.0, 70.0)
    altura = st.number_input("Altura (m)", 0.0, 2.5, 1.75)
    fumador = st.checkbox("Fumador")

    enviar = st.form_submit_button("Calcular riesgo", type="primary")

if enviar:
    riesgo = calcular(edad, peso, altura, fumador)
    st.metric("Riesgo estimado", f"{riesgo:.1%}")
```

**Cuándo usar formularios:**

- Cuando hay muchos campos y no quieres una re-ejecución por cada tecla.
- Cuando el cálculo es costoso y solo debe hacerse una vez con todos los datos.
- Cuando los valores deben validarse en conjunto.

**Reglas de los formularios:**

- Debe haber **exactamente un** `st.form_submit_button()` dentro.
- No se pueden usar `st.button()` normales dentro de un formulario.
- Los widgets dentro del formulario no disparan re-ejecuciones al cambiar.

## 7.6 Callbacks

Aunque el modelo general es de re-ejecución, los widgets aceptan un `on_change` (o `on_click` para botones) que se ejecuta **antes** de la re-ejecución:

```python
def registrar_cambio():
    st.session_state.log.append(f"Cambiado a {st.session_state.seleccion}")

if "log" not in st.session_state:
    st.session_state.log = []

st.selectbox("Opción", ["a", "b", "c"], key="seleccion", on_change=registrar_cambio)
st.write(st.session_state.log)
```

También admiten argumentos:

```python
def guardar(nombre, valor):
    st.session_state[nombre] = valor

st.button("Guardar", on_click=guardar, args=("clave", 42))
```

Los callbacks se usan poco en la práctica; la mayoría de casos se resuelven mejor con el flujo normal.

## 7.7 Control del flujo


### `st.stop()`

Detiene la ejecución del script en ese punto. Nada de lo que venga después se ejecuta ni se dibuja.

Es la herramienta clave para escribir código legible sin anidar condicionales:

```python
# En lugar de esto (difícil de seguir)
archivo = st.file_uploader("CSV")
if archivo is not None:
    df = pd.read_csv(archivo)
    if "target" in df.columns:
        if len(df) >= 10:
            entrenar(df)
        else:
            st.error("Muy pocas filas")
    else:
        st.error("Falta la columna target")
else:
    st.info("Sube un fichero")

# Esto (lineal y claro)
archivo = st.file_uploader("CSV")
if archivo is None:
    st.info("Sube un fichero para empezar")
    st.stop()

df = pd.read_csv(archivo)
if "target" not in df.columns:
    st.error(f"Falta la columna 'target'. Encontradas: {', '.join(df.columns)}")
    st.stop()

if len(df) < 10:
    st.error(f"Se necesitan al menos 10 filas, hay {len(df)}")
    st.stop()

entrenar(df)
```

Este patrón (comprobar y salir cuanto antes) hace el código mucho más mantenible.

### `st.rerun()`

Fuerza una re-ejecución inmediata. Se usa cuando has modificado `session_state` y necesitas que la interfaz se redibuje con los nuevos valores:

```python
if st.button("Reiniciar"):
    st.session_state.clear()
    st.rerun()
```

Úsalo con cuidado: un `st.rerun()` mal colocado provoca un bucle infinito.

## 7.8 Fragmentos: re-ejecutar solo una parte

Desde las versiones recientes, una función decorada con `@st.fragment` se re-ejecuta de forma aislada, **sin relanzar el resto del script**:

```python
@st.fragment
def panel_grafico():
    tipo = st.radio("Tipo de gráfico", ["Barras", "Líneas"], horizontal=True)
    st.pyplot(dibujar(tipo, datos))

# Esto tarda mucho, pero solo se ejecuta cuando cambia algo fuera del fragmento
datos = calculo_muy_lento()

panel_grafico()
```

Al cambiar el radio button, solo se vuelve a ejecutar `panel_grafico()`; `calculo_muy_lento()` no se toca. Es la alternativa a la caché cuando lo lento no se puede cachear.

También admite auto-refresco:

```python
@st.fragment(run_every="10s")
def panel_tiempo_real():
    st.metric("Última medida", leer_sensor())
```

---

# Módulo 8. Construcción guiada de una aplicación completa

Este módulo construye una aplicación paso a paso, añadiendo una capa en cada versión. La idea es que veas **cómo se llega** al código final, no solo el resultado.

El objetivo: una aplicación que carga un CSV, entrena un modelo de clasificación y permite hacer predicciones.


## 8.1 Versión 1: el esqueleto

Empezamos por lo mínimo que se puede ejecutar.

```python
import streamlit as st

st.set_page_config(page_title="Clasificador", layout="wide")

st.title("Clasificador automático")
st.write("Sube un CSV y entrena un modelo.")
```

Guárdalo como `app.py` y ejecútalo con `streamlit run app.py`. Debería verse un título y una línea de texto.

**Lo importante de esta versión:** `st.set_page_config()` es la primera llamada `st.*` del script. Si la pones después de cualquier otra, Streamlit lanza una excepción. Los `import` sí pueden ir antes.

## 8.2 Versión 2: cargar datos

```python
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Clasificador", layout="wide")

st.title("Clasificador automático")

# --- Carga de datos ---
archivo = st.file_uploader("Sube un fichero CSV", type=["csv"])

if archivo is None:
    st.info("Sube un fichero CSV para empezar.")
    st.stop()

df = pd.read_csv(archivo)

st.success(f"Cargado: {len(df)} filas, {len(df.columns)} columnas")
st.dataframe(df.head(), width="stretch")
```

**Lo importante de esta versión:** el patrón `if archivo is None: ... st.stop()`. Sin él, la línea `pd.read_csv(archivo)` fallaría al abrir la página, cuando todavía no hay fichero.


## 8.3 Versión 3: elegir las columnas

```python
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Clasificador", layout="wide")
st.title("Clasificador automático")

archivo = st.file_uploader("Sube un fichero CSV", type=["csv"])
if archivo is None:
    st.info("Sube un fichero CSV para empezar.")
    st.stop()

df = pd.read_csv(archivo)
st.success(f"Cargado: {len(df)} filas, {len(df.columns)} columnas")

with st.expander("Ver los datos"):
    st.dataframe(df, width="stretch")

# --- Configuración en la barra lateral ---
with st.sidebar:
    st.header("Configuración")

    objetivo = st.selectbox(
        "Columna objetivo",
        options=list(df.columns),
        index=len(df.columns) - 1,      # por defecto, la última
        help="La variable que quieres predecir",
    )

    predictoras = st.multiselect(
        "Variables predictoras",
        options=[c for c in df.columns if c != objetivo],
        default=[c for c in df.columns if c != objetivo],
    )

if not predictoras:
    st.warning("Selecciona al menos una variable predictora.")
    st.stop()

st.write(f"Predecir **{objetivo}** a partir de {len(predictoras)} variables.")
```

**Lo importante de esta versión:** las opciones del `selectbox` se generan a partir del fichero que el usuario acaba de subir. Esto resuelve el problema que tenía la versión de Gradio del cuaderno original, donde había que escribir a mano el nombre de la columna y se recibía un error si te equivocabas.

Fíjate también en cómo el `multiselect` excluye la columna objetivo de sus opciones: `[c for c in df.columns if c != objetivo]`.


## 8.4 Versión 4: entrenar el modelo

```python
import pandas as pd
import streamlit as st
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

st.set_page_config(page_title="Clasificador", layout="wide")
st.title("Clasificador automático")

# --- 1. Carga ---
archivo = st.file_uploader("Sube un fichero CSV", type=["csv"])
if archivo is None:
    st.info("Sube un fichero CSV para empezar.")
    st.stop()

@st.cache_data
def leer_csv(fichero):
    return pd.read_csv(fichero)

df = leer_csv(archivo)
st.success(f"Cargado: {len(df)} filas, {len(df.columns)} columnas")

with st.expander("Ver los datos"):
    st.dataframe(df, width="stretch")

# --- 2. Configuración ---
with st.sidebar:
    st.header("Configuración")
    objetivo = st.selectbox("Columna objetivo", list(df.columns),
                            index=len(df.columns) - 1)
    predictoras = st.multiselect(
        "Variables predictoras",
        [c for c in df.columns if c != objetivo],
        default=[c for c in df.columns if c != objetivo],
    )
    st.divider()
    n_arboles = st.slider("Nº de árboles", 10, 500, 100, 10)
    test_size = st.slider("% para test", 0.1, 0.5, 0.2, 0.05)

if not predictoras:
    st.warning("Selecciona al menos una variable predictora.")
    st.stop()

# --- 3. Entrenamiento ---
if st.button("Entrenar modelo", type="primary"):
    X = df[predictoras]
    y = df[objetivo]

    # Validación: solo aceptamos variables numéricas por simplicidad
    no_numericas = X.select_dtypes(exclude="number").columns.tolist()
    if no_numericas:
        st.error(f"Estas variables no son numéricas: {', '.join(no_numericas)}")
        st.stop()

    with st.spinner("Entrenando..."):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
        modelo = RandomForestClassifier(n_estimators=n_arboles, random_state=42)
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_test)
        acc = accuracy_score(y_test, pred)

    # Guardar en session_state para que sobreviva a la siguiente interacción
    st.session_state.modelo = modelo
    st.session_state.accuracy = acc
    st.session_state.predictoras = predictoras
    st.session_state.reporte = classification_report(y_test, pred, output_dict=True)

    st.success("Modelo entrenado")

# --- 4. Resultados ---
if "modelo" in st.session_state:
    st.divider()
    st.subheader("Resultados")

    c1, c2, c3 = st.columns(3)
    c1.metric("Accuracy", f"{st.session_state.accuracy:.3f}")
    c2.metric("Variables", len(st.session_state.predictoras))
    c3.metric("Árboles", st.session_state.modelo.n_estimators)

    with st.expander("Informe detallado por clase"):
        st.dataframe(pd.DataFrame(st.session_state.reporte).T, width="stretch")
```

**Lo importante de esta versión:** el uso de `st.session_state`. Si guardáramos el modelo en una variable normal, desaparecería en cuanto el usuario moviera cualquier control. Al guardarlo en `session_state`, la sección de resultados sigue mostrándose después.

Observa también la estructura: el bloque `if st.button(...)` **entrena y guarda**, y un bloque separado `if "modelo" in st.session_state` **muestra**. Separar ambas cosas es el patrón correcto.


## 8.5 Versión 5: predicción interactiva y descarga

Añadimos al final del fichero anterior:

```python
# --- 5. Predicción sobre nuevos valores ---
if "modelo" in st.session_state:
    st.divider()
    st.subheader("Predicción individual")

    with st.form("prediccion"):
        valores = {}
        columnas = st.columns(min(len(st.session_state.predictoras), 4))

        for i, variable in enumerate(st.session_state.predictoras):
            with columnas[i % len(columnas)]:
                valores[variable] = st.number_input(
                    variable,
                    value=float(df[variable].mean()),
                    key=f"pred_{variable}",
                )

        calcular = st.form_submit_button("Predecir", type="primary")

    if calcular:
        entrada = pd.DataFrame([valores])
        modelo = st.session_state.modelo

        prediccion = modelo.predict(entrada)[0]
        probabilidades = modelo.predict_proba(entrada)[0]
        confianza = probabilidades.max()

        c1, c2 = st.columns(2)
        c1.metric("Predicción", str(prediccion))
        c2.metric("Confianza", f"{confianza:.1%}")

        if confianza < 0.6:
            st.warning(
                "Confianza baja: estos valores caen en una zona donde el "
                "modelo no distingue bien entre clases."
            )

        tabla_probs = pd.DataFrame({
            "Clase": modelo.classes_,
            "Probabilidad": probabilidades.round(4),
        }).sort_values("Probabilidad", ascending=False)

        st.dataframe(tabla_probs, hide_index=True, width="stretch")

        # Descarga del resultado
        resultado = pd.DataFrame([{**valores, "prediccion": prediccion,
                                   "confianza": round(confianza, 4)}])
        st.download_button(
            "Descargar resultado (CSV)",
            data=resultado.to_csv(index=False).encode("utf-8"),
            file_name="prediccion.csv",
            mime="text/csv",
        )
```

**Lo importante de esta versión:**

- El **formulario** evita que se recalcule la predicción con cada tecla que el usuario escribe en los campos numéricos.
- Los campos se distribuyen automáticamente en columnas con `columnas[i % len(columnas)]`, lo que evita una lista interminable si hay muchas variables.
- Cada `number_input` lleva `key` única, obligatorio al crearlos dentro de un bucle.
- El valor inicial de cada campo es la **media de esa columna**, lo que da un punto de partida razonable.

## 8.6 Recapitulación de la estructura

El código final sigue este esquema, que sirve como plantilla para casi cualquier aplicación de ML:

```
1. Configuración de página          st.set_page_config()
2. Carga de datos                   file_uploader + cache_data + stop()
3. Configuración                    sidebar con widgets
4. Validación                       comprobaciones + stop()
5. Acción (entrenar)                button + session_state
6. Resultados                       if "modelo" in session_state
7. Interacción (predecir)           form + métricas
8. Exportación                      download_button
```

---

# Módulo 9. Publicación en internet con GitHub y Streamlit Community Cloud

Este módulo lleva la aplicación desde tu ordenador (o tu Colab) hasta una URL pública y permanente. Es el equivalente definitivo de todo lo que hemos hecho hasta ahora: nada de túneles temporales, nada de depender de que un cuaderno siga abierto.

## 9.1 Cómo funciona el sistema

Streamlit Community Cloud **no aloja tu código**: lo lee de GitHub. El flujo es este:

```
Tu ordenador  ──push──→  Repositorio GitHub  ──lee──→  Streamlit Cloud
                                                            │
                                                            ↓
                                              https://tu-app.streamlit.app
```

La consecuencia práctica más importante es que **cada vez que subas un cambio a GitHub, la aplicación se actualiza sola**. No hay que volver a desplegar nada.

Esto difiere de Hugging Face Spaces, donde el propio Space *era* el repositorio. Aquí son dos servicios distintos que se conectan.

## 9.2 Qué necesitas antes de empezar

1. Una cuenta de **GitHub** (gratuita) — [github.com/signup](https://github.com/signup)
2. Una cuenta en **Streamlit Community Cloud** — se crea con la de GitHub, no hace falta registro aparte
3. Los ficheros de tu aplicación

## 9.3 Los ficheros del repositorio, uno a uno

Vamos a generar cada fichero explicando qué hace y por qué es necesario.


### Estructura final

```
mi-app-streamlit/
├── streamlit_app.py          ← obligatorio: la aplicación
├── requirements.txt          ← obligatorio: las dependencias
├── README.md                 ← recomendado: documentación
├── .gitignore                ← recomendado: qué NO subir
└── .streamlit/
    └── config.toml           ← opcional: tema visual
```

### Fichero 1: `streamlit_app.py`

Es tu aplicación. El nombre no es obligatorio (puedes llamarlo `app.py`), pero `streamlit_app.py` es la convención que Streamlit Cloud detecta automáticamente, así que te ahorra un paso en la configuración.

**Tres cosas que debes revisar antes de subirlo:**

**a) Rutas relativas, nunca absolutas.**

```python
# MAL: esta ruta solo existe en tu ordenador
df = pd.read_csv("C:/Users/jose/Documentos/datos.csv")

# BIEN: relativa al repositorio
df = pd.read_csv("datos/datos.csv")

# MEJOR: robusta ante el directorio de trabajo
import os
RUTA = os.path.join(os.path.dirname(__file__), "datos", "datos.csv")
df = pd.read_csv(RUTA)
```

**b) Sin credenciales en el código.** Si tu aplicación necesita una clave de API, va en los *secrets* (sección 9.9), nunca escrita en el fichero.

**c) matplotlib con backend sin ventana.** Si usas matplotlib, añade esto antes de importar nada de matplotlib:

```python
import matplotlib
matplotlib.use("Agg")
from matplotlib.figure import Figure
```

Sin `Agg`, matplotlib intentará abrir una ventana gráfica en un servidor que no tiene pantalla, y la aplicación fallará.

### Fichero 2: `requirements.txt`

Lista de las librerías que Streamlit Cloud debe instalar. **Toda librería que importes debe estar aquí**, con la excepción de las de la biblioteca estándar de Python (`os`, `json`, `datetime`...).

Un ejemplo comentado:

```
streamlit>=1.40
scikit-learn>=1.4
pandas>=2.2
numpy>=1.26
matplotlib>=3.8
```

**Cómo generarlo correctamente.** Hay dos formas, y una de ellas es una trampa habitual:

```bash
# TRAMPA: incluye TODAS las librerías del entorno, cientos de líneas
pip freeze > requirements.txt
```

Si ejecutas `pip freeze` en tu entorno global de Python, obtendrás un fichero con doscientas librerías que tu aplicación no usa. Streamlit Cloud intentará instalarlas todas, tardará muchísimo y probablemente falle.

Solo es aceptable si lo haces **dentro de un entorno virtual limpio** creado para este proyecto (por eso lo recomendaba en el módulo 1):

```bash
source venv/bin/activate
pip freeze > requirements.txt
```

La alternativa más segura es **escribirlo a mano**, revisando los `import` de tu aplicación.

**Sobre fijar versiones: `==` frente a `>=`.**

Esta decisión importa más de lo que parece:

| Notación | Significado | Cuándo usarla |
|---|---|---|
| `scikit-learn>=1.4` | Cualquier versión desde la 1.4 | Por defecto. Más tolerante. |
| `scikit-learn==1.4.2` | Exactamente esa versión | Cuando cargas un modelo serializado |

La regla concreta: **si tu aplicación carga un fichero `.joblib` o `.pkl`, debes fijar la versión exacta con `==`**, y esa versión debe ser la misma con la que entrenaste el modelo. Un modelo serializado con una versión y cargado con otra puede dar avisos, errores o —lo peor— predicciones silenciosamente distintas.

Si en cambio tu aplicación entrena el modelo al arrancar (como el ejemplo del módulo 8), no hay ningún fichero atado a una versión y puedes usar `>=` con tranquilidad. Esto elimina toda una categoría de problemas de despliegue.


### Fichero 3: `README.md`

Documentación normal en Markdown. A diferencia de Hugging Face Spaces, **aquí no lleva cabecera YAML**: es un README corriente.

```markdown
# Clasificador automático

Aplicación que entrena un Random Forest sobre un CSV subido por el usuario
y permite hacer predicciones interactivas.

## Ejecutar en local

```bash
pip install -r requirements.txt
streamlit run streamlit_app.py
```

## Uso

1. Sube un fichero CSV con datos numéricos.
2. Selecciona la columna objetivo y las predictoras.
3. Pulsa "Entrenar modelo".
4. Introduce valores nuevos para obtener una predicción.
```

### Fichero 4: `.gitignore`

Indica a Git qué ficheros **no** debe subir. Es importante por dos razones: evita subir basura, y evita subir secretos por accidente.

```
# Entornos virtuales
venv/
.venv/
env/

# Caché de Python
__pycache__/
*.pyc

# Secretos: NUNCA deben llegar a GitHub
.streamlit/secrets.toml

# Sistema operativo
.DS_Store
Thumbs.db

# Editores
.vscode/
.idea/
```

La línea de `secrets.toml` es la más importante. Si subes ese fichero a un repositorio público, tus claves quedan expuestas.

### Fichero 5: `.streamlit/config.toml` (opcional)

Personaliza el aspecto de la aplicación:

```toml
[theme]
primaryColor = "#2563eb"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f1f5f9"
textColor = "#0f172a"
font = "sans serif"

[server]
maxUploadSize = 50
```

`primaryColor` afecta a botones, sliders y elementos destacados. `maxUploadSize` está en megabytes y por defecto es 200.

Fíjate en que va dentro de una carpeta llamada `.streamlit` (con punto delante). En algunos sistemas las carpetas que empiezan por punto están ocultas por defecto en el explorador de archivos.

## 9.4 Subir a GitHub: método A, por navegador

Este método no requiere instalar nada. Es el recomendado si no has usado Git antes.

**Paso 1: crear el repositorio**

Ve a [github.com/new](https://github.com/new) y rellena:

- **Repository name**: `mi-app-streamlit` (sin espacios; usa guiones)
- **Description**: opcional
- **Public** — importante: si lo haces privado necesitarás dar permisos adicionales a Streamlit
- **NO marques** "Add a README file" (vas a subir el tuyo)

Pulsa **Create repository**.

**Paso 2: subir los ficheros**

En la página que aparece, busca el enlace **"uploading an existing file"**, o ve a **Add file → Upload files**.

Arrastra `streamlit_app.py`, `requirements.txt`, `README.md` y `.gitignore`.

> **Problema con la carpeta `.streamlit`:** el navegador no siempre permite arrastrar carpetas que empiezan por punto. Si te ocurre, créala después con **Add file → Create new file** y escribe en el campo del nombre: `.streamlit/config.toml`. GitHub crea la carpeta automáticamente al detectar la barra.

**Paso 3: confirmar**

Abajo, escribe un mensaje de commit (por ejemplo, "Primera versión") y pulsa **Commit changes**.

**Paso 4: verificar**

Comprueba en la pestaña **Code** que aparecen todos los ficheros. Fíjate especialmente en que `requirements.txt` no esté vacío.

## 9.5 Subir a GitHub: método B, por línea de comandos

Más ágil una vez configurado, y el que usarás para actualizar la aplicación.

**Paso 1: instalar y configurar Git** (solo la primera vez)

Descarga Git de [git-scm.com](https://git-scm.com/downloads) y configura tu identidad:

```bash
git config --global user.name "Tu Nombre"
git config --global user.email "tu@email.com"
```

**Paso 2: crear el repositorio vacío en GitHub**

Igual que en el paso 1 del método A, en [github.com/new](https://github.com/new). Anota la URL que te da, del tipo `https://github.com/TU_USUARIO/mi-app-streamlit.git`.

**Paso 3: inicializar y subir**

Desde la carpeta de tu proyecto:

```bash
# Inicializar el repositorio local
git init

# Añadir todos los ficheros (respetando .gitignore)
git add .

# Ver qué se va a subir (comprobación recomendada)
git status

# Crear el commit
git commit -m "Primera versión de la aplicación"

# Renombrar la rama a 'main' (convención actual)
git branch -M main

# Conectar con GitHub
git remote add origin https://github.com/TU_USUARIO/mi-app-streamlit.git

# Subir
git push -u origin main
```

En el `git push` te pedirá credenciales. **La contraseña de GitHub ya no funciona**: necesitas un *Personal Access Token*. Se genera en:

```
GitHub → foto de perfil → Settings → Developer settings
       → Personal access tokens → Tokens (classic) → Generate new token
```

Marca el permiso **`repo`**, genera el token y **cópialo inmediatamente** (no se puede volver a ver). Úsalo como contraseña cuando Git te la pida.

**Paso 4: verificar**

Recarga la página del repositorio en GitHub. Deberían aparecer tus ficheros.

## 9.6 Desplegar en Streamlit Community Cloud

Ahora conectamos el repositorio con la plataforma.

**Paso 1: entrar**

Ve a [share.streamlit.io](https://share.streamlit.io) y pulsa **Continue with GitHub**.

La primera vez, GitHub te pedirá autorizar a Streamlit. Necesita permiso para leer tus repositorios y para gestionar las claves de despliegue.

**Paso 2: crear la aplicación**

Pulsa **Create app** (o **New app** según la versión de la interfaz).

Elige la opción **Deploy a public app from GitHub**.

**Paso 3: rellenar el formulario**

Tres campos:

- **Repository**: empieza a escribir el nombre y se autocompleta → `TU_USUARIO/mi-app-streamlit`
- **Branch**: `main`
- **Main file path**: `streamlit_app.py`

Y uno opcional pero recomendable:

- **App URL**: el subdominio que quieras. Solo admite letras, números y guiones. Si lo dejas vacío, Streamlit genera uno aleatorio poco memorable.

**Paso 4: Advanced settings (no te lo saltes)**

Despliega **Advanced settings** y presta atención a este campo:

- **Python version**: elige **3.11** o superior.

> **Este paso merece un aviso serio.** La versión de Python **no se puede cambiar una vez desplegada la aplicación**. Si necesitas cambiarla, hay que borrar la app y volver a crearla desde cero.
>
> Y es un problema real: muchas librerías modernas ya no publican versiones para Python 3.9 o 3.10. Si dejas una versión antigua y tu `requirements.txt` pide algo reciente, obtendrás un error de instalación del tipo *"Could not find a version that satisfies the requirement"*, y la línea que lo explica (*"Ignored the following versions that require a different python version"*) pasa fácilmente desapercibida.

Aquí también puedes pegar los secretos, si tu aplicación los necesita (sección 9.9).

**Paso 5: desplegar**

Pulsa **Deploy**.

Verás una pantalla con el log de construcción en tiempo real. El proceso es:

1. Clonar el repositorio
2. Crear el entorno de Python
3. Instalar `requirements.txt`
4. Arrancar la aplicación

Tarda entre dos y cinco minutos la primera vez. Cuando termine, tu aplicación estará en:

```
https://NOMBRE-QUE-ELEGISTE.streamlit.app
```

Esa URL es **pública y permanente**. Cualquiera puede abrirla sin cuenta de Streamlit ni de GitHub.


## 9.7 Actualizar la aplicación

No hay que hacer nada especial. **Cada `git push` actualiza la aplicación automáticamente.**

El ciclo de trabajo habitual:

```bash
# 1. Editar el código y probar en local
streamlit run streamlit_app.py

# 2. Cuando funcione, subir
git add .
git commit -m "Añade gráfico de importancia de variables"
git push

# 3. Esperar un minuto: la app se actualiza sola
```

Si has subido los ficheros por navegador, cada **Commit changes** produce el mismo efecto.

Un límite a conocer: Community Cloud limita las actualizaciones desde GitHub a **cinco por minuto**. Si haces muchos commits seguidos, algunos se ignorarán.

## 9.8 Gestionar la aplicación desplegada

Desde [share.streamlit.io](https://share.streamlit.io) verás la lista de tus aplicaciones. En el menú de tres puntos de cada una:

| Opción | Qué hace |
|---|---|
| **Settings** | Cambiar secretos, ver la configuración |
| **Reboot app** | Reiniciar sin cambiar el código (útil si se ha quedado colgada) |
| **Delete app** | Borrar el despliegue (el repositorio de GitHub no se toca) |
| **Analytics** | Ver el número de visitas |

Y dentro de la propia aplicación, abajo a la derecha, hay un botón **Manage app** que despliega los **logs en vivo**. Es donde verás los errores de ejecución. El equivalente a los *Container logs* de Hugging Face.

## 9.9 Secretos y credenciales

Si tu aplicación necesita una clave de API, nunca la escribas en el código de un repositorio público.

**En la nube:** panel de la aplicación → **Settings → Secrets**. El formato es TOML:

```toml
API_KEY = "abc123xyz"
MODELO_URL = "https://ejemplo.com/modelo"

[base_datos]
host = "db.ejemplo.com"
usuario = "admin"
password = "secreto"
```

**En el código:**

```python
clave = st.secrets["API_KEY"]
host = st.secrets["base_datos"]["host"]
```

**En local:** crea el fichero `.streamlit/secrets.toml` con el mismo contenido. Y asegúrate de que está en `.gitignore` (ya lo incluimos en 9.3).


## 9.10 Límites de la capa gratuita

| Aspecto | Límite |
|---|---|
| Coste | Gratuito |
| Memoria por aplicación | En torno a **1 GB de RAM** |
| Repositorio | GitHub, público o privado |
| Hibernación | Las apps sin visitas se duermen; despiertan al recibir una |
| Actualizaciones | Máximo 5 por minuto |
| Sistema base | Debian Linux |

**El límite de memoria es el que más suele doler.** 1 GB es cómodo para scikit-learn con datasets moderados, pero insuficiente para modelos de deep learning. Si tu aplicación se cae sin mensaje claro, la falta de memoria es el primer sospechoso.

Sobre la hibernación: no se pierde nada, pero el primer visitante tras un periodo de inactividad esperará unos segundos mientras la aplicación se reinicia. Si vas a enseñarla en una presentación, ábrela unos minutos antes.

## 9.11 Diagnóstico de errores en el despliegue

Antes de nada, **la regla de oro**: prueba en local con un entorno virtual limpio.

```bash
python -m venv test_env
source test_env/bin/activate
pip install -r requirements.txt
streamlit run streamlit_app.py
```

Si funciona así, funcionará desplegado. La mayoría de fallos vienen de librerías que tienes instaladas en tu ordenador pero que olvidaste poner en `requirements.txt`.

### Errores durante la instalación

| Mensaje | Causa | Solución |
|---|---|---|
| `ModuleNotFoundError: No module named 'X'` | Falta en `requirements.txt` | Añádelo y haz push |
| `Could not find a version that satisfies the requirement` | La versión no existe para el Python elegido | Ver el aviso del paso 4 (9.6) |
| `Ignored the following versions that require a different python version` | Lo mismo. Esta línea es la pista real. | Borra la app y redespliégala con Python 3.11+ |
| La instalación tarda muchísimo | Dependencias pesadas o `pip freeze` completo | Reduce `requirements.txt` a lo imprescindible |

### Errores durante la ejecución

| Mensaje | Causa | Solución |
|---|---|---|
| `FileNotFoundError` | Ruta absoluta, o el fichero no se subió | Usa rutas relativas; verifica en GitHub que el fichero está |
| `StreamlitAPIException: set_page_config()` | No es la primera llamada `st.*` | Muévela justo después de los imports |
| `DuplicateWidgetID` | Dos widgets idénticos | Añade `key="algo_unico"` |
| La app se cae sin mensaje | Falta de memoria (límite de 1 GB) | Reduce el modelo o los datos |
| Va lentísima | Falta caché | Añade `@st.cache_data` / `@st.cache_resource` |
| Los cambios no aparecen | Push a otra rama | Verifica que la rama desplegada es la correcta |

## 9.12 Comparación con Hugging Face Spaces

Por si te preguntas cuál elegir:

| | Streamlit Community Cloud | Hugging Face Spaces |
|---|---|---|
| Dónde vive el código | GitHub | El propio Space |
| Configuración | Panel web + `config.toml` | Cabecera YAML del `README.md` |
| Versión de Python | Se elige al desplegar, **no se puede cambiar después** | Campo `python_version` en el YAML |
| Memoria | ~1 GB | 16 GB en CPU Basic |
| Frameworks | Streamlit | Gradio, Docker, estático |
| Plan gratuito | Sí | Conviene verificar las condiciones actuales |

---

# Módulo 10. Buenas prácticas y catálogo de errores

## 10.1 Estructura del código

**Separa la lógica de la interfaz.** Las funciones que calculan no deberían importar Streamlit:

```python
# logica.py  (testeable con pytest, sin arrancar ningún servidor)
def calcular_riesgo(edad, peso, altura, fumador):
    imc = peso / altura ** 2
    riesgo = 0.01 * edad + 0.05 * max(0, imc - 25)
    if fumador:
        riesgo *= 1.8
    return min(riesgo, 1.0)

# streamlit_app.py
from logica import calcular_riesgo
```

**Ordena el script de forma predecible.** Este esqueleto funciona para casi cualquier aplicación:

```python
# 1. Imports
# 2. st.set_page_config()
# 3. Constantes
# 4. Funciones cacheadas (carga de datos y modelos)
# 5. Funciones de lógica
# 6. Barra lateral (configuración)
# 7. Validaciones + st.stop()
# 8. Cuerpo principal
```

## 10.2 Rendimiento

- **Cachea todo lo que tarde más de un segundo.** Es la optimización con mayor retorno.
- **Usa formularios** cuando haya muchos campos: evitas una re-ejecución por tecla.
- **Usa `@st.fragment`** para paneles que se refrescan sin relanzar todo.
- **Filtra los datos antes de mostrarlos.** Con 1 GB de límite, no cargues DataFrames enormes.
- **Evita `pyplot` global.** Usa la clase `Figure` para no acumular figuras.


## 10.3 Experiencia de usuario

- Controles en la **barra lateral**, resultados en el área principal.
- `st.metric` para el resultado principal: destaca visualmente.
- `help="..."` en cada widget que no sea obvio.
- `st.spinner` en toda operación que tarde más de un segundo.
- **Explica las limitaciones del modelo.** Si se entrenó con 150 muestras, dilo.
- Ofrece **descarga de resultados** con `st.download_button`.
- Mensajes de error **accionables**: no "Error en los datos", sino "Falta la columna 'target'. Encontradas: a, b, c".


## 10.4 Los diez errores más frecuentes

**1. Ejecutar con `python` en vez de `streamlit run`.** No da error claro; simplemente no funciona.

**2. Olvidar la caché.** La aplicación funciona pero es insoportablemente lenta.

**3. `st.set_page_config()` en el sitio equivocado.** Debe ser la primera llamada `st.*` del script.

**4. Esperar que `st.button()` recuerde su estado.** Solo es `True` en la re-ejecución inmediata al clic.

**5. Widgets duplicados sin `key`.** Error `DuplicateWidgetID`, típico dentro de bucles.

**6. Rutas absolutas.** Funcionan en tu ordenador y fallan en el servidor.

**7. `pip freeze` desde el entorno global.** Genera un `requirements.txt` de cientos de líneas.

**8. Elegir mal la versión de Python al desplegar.** No se puede cambiar después.

**9. Usar `pyplot` en lugar de `Figure`.** Fuga de memoria progresiva.

**10. Subir `secrets.toml` a GitHub.** Expone tus credenciales públicamente.

---

# Apéndice A. Chuleta de referencia

## Esqueleto

```python
import streamlit as st
import pandas as pd

st.set_page_config(page_title="Mi app", layout="wide")   # SIEMPRE primero

@st.cache_resource
def cargar_modelo():
    return joblib.load("modelo.joblib")

modelo = cargar_modelo()

with st.sidebar:
    parametro = st.slider("Parámetro", 0, 100, 50)

st.title("Mi aplicación")
st.metric("Resultado", modelo.predict([[parametro]])[0])
```

## Entradas

```python
st.text_input()      st.text_area()       st.number_input()
st.slider()          st.select_slider()   st.checkbox()
st.radio()           st.selectbox()       st.multiselect()
st.date_input()      st.time_input()      st.color_picker()
st.file_uploader()   st.camera_input()    st.data_editor()
st.button()          st.download_button() st.form_submit_button()
```

## Salidas

```python
st.title()      st.header()     st.subheader()   st.markdown()
st.write()      st.caption()    st.code()        st.latex()
st.dataframe()  st.table()      st.metric()      st.json()
st.pyplot()     st.plotly_chart()   st.line_chart()   st.bar_chart()
st.image()      st.audio()      st.video()
st.success()    st.info()       st.warning()     st.error()
```

## Layout

```python
with st.sidebar: ...
col1, col2 = st.columns([2, 1])
tab1, tab2 = st.tabs(["A", "B"])
with st.expander("Más"): ...
with st.container(border=True): ...
hueco = st.empty()
```

## Caché

```python
@st.cache_data        # DataFrames, listas, dicts (copia por usuario)
@st.cache_resource    # modelos, conexiones (objeto compartido)

@st.cache_data(ttl=3600, max_entries=10, show_spinner="Cargando...")

mi_funcion.clear()
```

## Estado y flujo

```python
if "clave" not in st.session_state:
    st.session_state.clave = valor_inicial

st.session_state.clave = nuevo
st.stop()        # detener el script aquí
st.rerun()       # re-ejecutar inmediatamente

with st.form("f"):
    ...
    enviar = st.form_submit_button("Enviar")
```

## Progreso

```python
with st.spinner("Cargando..."): ...
barra = st.progress(0.0, text="...")
with st.status("Proceso", expanded=True) as s:
    s.update(label="Hecho", state="complete")
```

## Comandos

```bash
streamlit run app.py
streamlit run app.py --server.port 8502
streamlit run app.py --server.address 0.0.0.0
streamlit run app.py --server.headless true
streamlit --version
streamlit hello
```

## Colab con ngrok

```python
!pip install -q streamlit pyngrok

# %%writefile app.py  (en una celda aparte)

from pyngrok import ngrok
import subprocess, time

ngrok.set_auth_token("TU_TOKEN")
ngrok.kill()
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)
print(ngrok.connect(8501))
```

---

# Apéndice B. Equivalencias con Gradio

| Gradio | Streamlit |
|---|---|
| `gr.Textbox()` | `st.text_input()` / `st.text_area()` |
| `gr.Number()` | `st.number_input()` |
| `gr.Slider()` | `st.slider()` |
| `gr.Checkbox()` | `st.checkbox()` |
| `gr.Radio()` | `st.radio()` |
| `gr.Dropdown()` | `st.selectbox()` / `st.multiselect()` |
| `gr.File()` | `st.file_uploader()` |
| `gr.Dataframe()` | `st.dataframe()` / `st.data_editor()` |
| `gr.Label()` | `st.metric()` + `st.dataframe()` |
| `gr.Plot()` | `st.pyplot()` / `st.plotly_chart()` |
| `gr.Markdown()` | `st.markdown()` |
| `gr.Button()` | `st.button()` |
| `gr.Row()` | `st.columns()` |
| `gr.Column()` | `with col:` |
| `gr.Tab()` | `st.tabs()` |
| `gr.Accordion()` | `st.expander()` |
| `gr.State()` | `st.session_state` |
| `raise gr.Error()` | `st.error()` + `st.stop()` |
| `gr.Warning()` | `st.warning()` |
| `gr.Info()` | `st.info()` |
| `gr.Progress()` | `st.progress()` / `st.spinner()` |
| `gr.Examples()` | `st.selectbox()` con valores predefinidos |
| carga fuera de la función | `@st.cache_resource` |
| `demo.launch()` | `streamlit run app.py` |

## Diferencias de fondo

**Reactividad.** En Gradio hay que conectar `.change()` para que un control recalcule. En Streamlit es el comportamiento por defecto.

**Ficheros subidos.** Gradio devuelve un objeto del que hay que sacar `.name`; Streamlit devuelve algo que pandas lee directamente.

**Carga de modelos.** En Gradio basta con cargarlo fuera de la función. En Streamlit **hace falta caché**, porque no hay "fuera de la función": todo el script se re-ejecuta.

**Ejecución.** Gradio: `python app.py`. Streamlit: `streamlit run app.py`.

---

## Recursos

- Documentación: `docs.streamlit.io`
- Referencia de la API: `docs.streamlit.io/develop/api-reference`
- Galería de ejemplos: `streamlit.io/gallery`
- Community Cloud: `share.streamlit.io`
- Foro: `discuss.streamlit.io`
- Estado del servicio: `streamlitstatus.com`
- ngrok: `dashboard.ngrok.com`
